In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "3"

try:
    from google.protobuf import message_factory

    if not hasattr(message_factory.MessageFactory, "GetPrototype"):

        def _GetPrototype(self, descriptor):
            return self.GetMessageClass(descriptor)

        message_factory.MessageFactory.GetPrototype = _GetPrototype
except Exception:
    pass  # If protobuf isn’t present, TF will raise its own error later

import tempfile

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy("mixed_float16")

tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(4)

ROOT_DIR = "/kaggle/input"
DATA_DIR = os.path.join(ROOT_DIR, "cassava-leaf-disease-classification")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train_images")
TEST_IMG_DIR = os.path.join(DATA_DIR, "test_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
SAMPLE_SUBMIT = os.path.join(DATA_DIR, "sample_submission.csv")

HEIGHT, WIDTH = 240, 240  # EfficientNetB0 input size
BATCH_SIZE = 32
NUM_CLASSES = 5
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

tf.random.set_seed(SEED)
np.random.seed(SEED)

train_df = pd.read_csv(TRAIN_CSV)
sample_sub_df = pd.read_csv(SAMPLE_SUBMIT)

train_paths = [os.path.join(TRAIN_IMG_DIR, img_id) for img_id in train_df["image_id"]]
train_labels = train_df["label"].values.astype(np.int32)

test_ids = sample_sub_df["image_id"].values
test_paths = [os.path.join(TEST_IMG_DIR, img_id) for img_id in test_ids]


def _decode_and_resize(path):
    """Read an image file, decode JPEG, convert to float32, resize, and preprocess for EfficientNet."""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # scales to [0,1]
    img = tf.image.resize(img, [HEIGHT, WIDTH])
    # EfficientNet expects inputs in the range [-1, 1]
    img = tf.keras.applications.efficientnet.preprocess_input(img * 255.0)
    return img


def _augment(image, label):
    """Simple augmentation: random flips, random 90° rotations, random brightness, and random zoom."""
    image = tf.image.random_flip_left_right(image)  # no fixed seed for more variability
    image = tf.image.random_flip_up_down(image)
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    image = tf.image.random_brightness(image, max_delta=0.1)
    # Random zoom via central crop + resize
    scales = tf.random.uniform([], 0.9, 1.0)
    new_height = tf.cast(scales * tf.cast(tf.shape(image)[0], tf.float32), tf.int32)
    new_width = tf.cast(scales * tf.cast(tf.shape(image)[1], tf.float32), tf.int32)
    image = tf.image.random_crop(image, size=[new_height, new_width, 3])
    image = tf.image.resize(image, [HEIGHT, WIDTH])
    return image, label


def _load_test(path):
    """Decode and resize a test image (no label)."""
    img = _decode_and_resize(path)
    return img




2026-05-02 09:23:16.409301: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777713796.422553      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777713796.426639      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 09:23:16.442840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
cache_dir = tempfile.gettempdir()
train_cache_path = os.path.join(cache_dir, "train_cache")
train_ds_raw = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_ds_raw = train_ds_raw.map(
    lambda p, l: (_decode_and_resize(p), l), num_parallel_calls=AUTOTUNE
).cache(train_cache_path)

train_dataset = train_ds_raw.map(_augment, num_parallel_calls=AUTOTUNE)
train_dataset = (
    train_dataset.shuffle(10000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)
)




2026-05-02 09:23:23.483456: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777713803.483893      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [3]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(HEIGHT, WIDTH, 3)
)

base_model.trainable = False

inputs = tf.keras.layers.Input(shape=(HEIGHT, WIDTH, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

# Slightly longer training schedules (12 epochs each phase)
EPOCHS_PHASE1 = 12
EPOCHS_PHASE2 = 12

model.fit(train_dataset, epochs=EPOCHS_PHASE1, verbose=2)

base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

model.fit(train_dataset, epochs=EPOCHS_PHASE2, verbose=2)




16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/12


2026-05-02 09:23:45.644546: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 6354 of 10000
2026-05-02 09:23:51.501576: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.
I0000 00:00:1777713831.607941     107 service.cc:148] XLA service 0x7f156c002640 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777713831.607968     107 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-02 09:23:51.835407: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777713832.704074     107 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-02 09:23:53.881834: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in fun

586/586 - 58s - 100ms/step - accuracy: 0.7009 - loss: 0.8209
Epoch 2/12
586/586 - 11s - 18ms/step - accuracy: 0.7390 - loss: 0.7163
Epoch 3/12
586/586 - 11s - 20ms/step - accuracy: 0.7485 - loss: 0.6866
Epoch 4/12
586/586 - 11s - 19ms/step - accuracy: 0.7512 - loss: 0.6780
Epoch 5/12
586/586 - 12s - 20ms/step - accuracy: 0.7546 - loss: 0.6650
Epoch 6/12
586/586 - 11s - 19ms/step - accuracy: 0.7613 - loss: 0.6585
Epoch 7/12
586/586 - 11s - 18ms/step - accuracy: 0.7585 - loss: 0.6535
Epoch 8/12
586/586 - 11s - 19ms/step - accuracy: 0.7611 - loss: 0.6497
Epoch 9/12
586/586 - 11s - 18ms/step - accuracy: 0.7619 - loss: 0.6499
Epoch 10/12
586/586 - 11s - 18ms/step - accuracy: 0.7634 - loss: 0.6470
Epoch 11/12
586/586 - 11s - 18ms/step - accuracy: 0.7661 - loss: 0.6434
Epoch 12/12
586/586 - 12s - 20ms/step - accuracy: 0.7658 - loss: 0.6428
Epoch 1/12
586/586 - 39s - 67ms/step - accuracy: 0.7394 - loss: 0.7173
Epoch 2/12
586/586 - 11s - 19ms/step - accuracy: 0.7949 - loss: 0.5702
Epoch 3/12
58

In [4]:
test_cache_path = os.path.join(cache_dir, "test_cache")
test_ds = tf.data.Dataset.from_tensor_slices(test_paths)
test_ds = test_ds.map(_load_test, num_parallel_calls=AUTOTUNE).cache(test_cache_path)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

pred_probs = model.predict(test_ds, verbose=0)
pred_labels = np.argmax(pred_probs, axis=1)

submission_df = pd.DataFrame({"image_id": test_ids, "label": pred_labels})
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}, rows: {len(submission_df)}")

2026-05-02 09:29:18.442075: W tensorflow/core/kernels/data/cache_dataset_ops.cc:332] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-05-02 09:29:23.444079: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2276', 996 bytes spill stores, 820 bytes spill loads

2026-05-02 09:29:23.643825: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2276', 1376 bytes spill stores, 1272 bytes spill loads

E0000 00:00:1777714164.868965     105 gpu_timer.cc:82] Delay kernel timed out: measured time has sub

Submission written to submission.csv, rows: 2676
